# Portfolio Optimizer — From Markowitz to Black-Litterman

**Objective:** Build an optimized multi-asset portfolio using three complementary methods, backtest the strategy, and analyze risk metrics.

This notebook demonstrates:
1. Fetching market data and computing Ledoit-Wolf shrinkage covariance
2. **Fama-French 6-Factor expected returns** — an alternative to noisy historical means
3. Mean-Variance optimization (maximum Sharpe ratio)
4. Efficient frontier construction
5. Risk Parity allocation (equal risk contribution)
6. Black-Litterman with analyst views
7. Walk-forward backtesting with periodic rebalancing
8. Risk metrics comparison (optimized vs equal-weight vs best asset)

In [ ]:
# Colab setup — run this cell if you are on Google Colab
import os
if "COLAB_RELEASE_TAG" in os.environ:
    !git clone --depth 1 https://github.com/louisgay/quant-apps.git /content/quant-apps
    !pip install -q -r /content/quant-apps/portfolio_optimizer/requirements.txt
    os.chdir("/content/quant-apps/portfolio_optimizer")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

sys.path.insert(0, str(Path.cwd()))

from engine import (
    MarketData, fetch_market_data, fetch_ff_factors, estimate_expected_returns,
    MeanVarianceOptimizer, RiskParityOptimizer, BlackLittermanOptimizer,
    PortfolioMetrics, Backtester, rolling_sharpe, drawdown_series,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 11})

print("Engine loaded successfully.")

---
## 1. Market Data

We fetch 5 years of daily prices for a diversified basket of 5 US equities via Yahoo Finance. The covariance matrix is estimated using **Ledoit-Wolf shrinkage** — a robust alternative to the sample covariance that avoids overfitting in high dimensions.

In [ ]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "JPM", "XOM"]

data = fetch_market_data(TICKERS, lookback_years=5, risk_free_rate=0.04)

print(f"Assets: {data.tickers}")
print(f"Observations: {len(data.returns)} trading days")
print(f"Risk-free rate: {data.risk_free_rate:.1%}")
print(f"\nAnnualized expected returns:")
for t, r in zip(data.tickers, data.expected_returns):
    print(f"  {t:6s} {r:+.2%}")

In [ ]:
# Correlation heatmap
corr = data.correlation_matrix
corr_df = pd.DataFrame(corr, index=data.tickers, columns=data.tickers)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
axes[0].set_xticks(range(len(data.tickers)))
axes[0].set_yticks(range(len(data.tickers)))
axes[0].set_xticklabels(data.tickers)
axes[0].set_yticklabels(data.tickers)
for i in range(len(data.tickers)):
    for j in range(len(data.tickers)):
        axes[0].text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=10)
axes[0].set_title("Correlation Matrix", fontweight="bold")
plt.colorbar(im, ax=axes[0], fraction=0.046)

# Return vs volatility scatter
axes[1].scatter(data.annualized_vols * 100, data.expected_returns * 100,
                s=100, zorder=5, color="steelblue", edgecolor="white")
for i, t in enumerate(data.tickers):
    axes[1].annotate(t, (data.annualized_vols[i] * 100, data.expected_returns[i] * 100),
                     textcoords="offset points", xytext=(8, 4), fontweight="bold")
axes[1].set_xlabel("Annualized Volatility (%)")
axes[1].set_ylabel("Annualized Return (%)")
axes[1].set_title("Risk-Return Profile", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 1b. Fama-French 6-Factor Expected Returns

Historical mean returns ($\bar{r}_i \cdot 252$) are **noisy** and tend to produce overfitted allocations. An alternative is the **Fama-French 6-factor model**: regress each asset's excess returns on 6 compensated risk factors, then use the fitted model as the expected return estimate.

For each asset $i$, we run the time-series regression:

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i^{\text{Mkt}} \cdot \text{MktRF}_t + \beta_i^{\text{SMB}} \cdot \text{SMB}_t + \beta_i^{\text{HML}} \cdot \text{HML}_t + \beta_i^{\text{RMW}} \cdot \text{RMW}_t + \beta_i^{\text{CMA}} \cdot \text{CMA}_t + \beta_i^{\text{Mom}} \cdot \text{Mom}_t + \varepsilon_{i,t}$$

The annualized expected return is then:

$$\hat{\mu}_i = \left(\alpha_i + \boldsymbol{\beta}_i^+ \cdot \overline{\mathbf{f}}\right) \times 252 + \bar{r}_f \times 252$$

where $\boldsymbol{\beta}_i^+ = \max(\boldsymbol{\beta}_i, 0)$ floors all betas at zero — negative exposure to a compensated risk premium is economically irrational for a long portfolio.

In [ ]:
# Fetch Fama-French 6 factors
import pandas as pd

start = pd.Timestamp.now() - pd.DateOffset(years=5)
end = pd.Timestamp.now()
ff = fetch_ff_factors(start, end)

print(f"FF factors: {list(ff.columns)}")
print(f"Date range: {ff.index.min().date()} to {ff.index.max().date()}")
print(f"Observations: {len(ff)}")

# Estimate expected returns using both methods
mu_hist = estimate_expected_returns(data.returns, "historical")
mu_ff6 = estimate_expected_returns(data.returns, "ff6", ff)

# Compare
comp_df = pd.DataFrame({
    "Historical Mean": mu_hist,
    "FF6 Model": mu_ff6,
    "Difference": mu_ff6 - mu_hist,
}, index=data.tickers)

display(comp_df.style.format("{:.2%}"))

# Visual comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(data.tickers))
width = 0.35

bars1 = ax.bar(x - width/2, mu_hist * 100, width, label="Historical Mean",
               color="#90CAF9", edgecolor="white")
bars2 = ax.bar(x + width/2, mu_ff6 * 100, width, label="FF6 Model",
               color="#1565C0", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(data.tickers)
ax.set_ylabel("Annualized Expected Return (%)")
ax.set_title("Expected Returns: Historical Mean vs Fama-French 6-Factor", fontweight="bold")
ax.legend()
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

---
## 2. Mean-Variance Optimization

The Markowitz framework maximizes the **Sharpe ratio**:

$$\max_w \frac{w^T \mu - r_f}{\sqrt{w^T \Sigma w}} \quad \text{s.t.} \quad \sum w_i = 1, \; w_i \geq 0$$

Solved via SLSQP (Sequential Least-Squares Programming).

In [ ]:
mv = MeanVarianceOptimizer(data, long_only=True, max_weight=0.40)
opt = mv.max_sharpe()

print("="*55)
print("       MAX SHARPE PORTFOLIO")
print("="*55)
print(f"  Expected Return:  {opt.expected_return:>10.2%}")
print(f"  Volatility:       {opt.volatility:>10.2%}")
print(f"  Sharpe Ratio:     {opt.sharpe_ratio:>10.2f}")
print(f"\n  Weights:")
for t, w in zip(data.tickers, opt.weights):
    print(f"    {t:6s} {w:>8.1%}")
print("="*55)

In [ ]:
# Weights bar chart
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(data.tickers, opt.weights * 100, color=colors, edgecolor="white")
for bar, w in zip(bars, opt.weights):
    if w > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{w:.1%}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Weight (%)")
ax.set_title("Max Sharpe Portfolio — Optimal Weights", fontweight="bold")
ax.set_ylim(0, max(opt.weights) * 120)
plt.tight_layout()
plt.show()

---
## 3. Efficient Frontier

The efficient frontier traces all portfolios that maximize return for a given level of risk. We compute it by sweeping target returns from the minimum-variance portfolio to the maximum achievable return.

In [ ]:
frontier = mv.efficient_frontier(n_points=50)

f_vols = [p.volatility * 100 for p in frontier]
f_rets = [p.expected_return * 100 for p in frontier]

fig, ax = plt.subplots(figsize=(10, 6))

# Frontier curve
ax.plot(f_vols, f_rets, "b-", linewidth=2, label="Efficient Frontier")

# Optimal portfolio
ax.scatter(opt.volatility * 100, opt.expected_return * 100,
           marker="*", s=300, color="gold", edgecolor="black", zorder=10,
           label=f"Max Sharpe (SR={opt.sharpe_ratio:.2f})")

# Equal-weight portfolio
ew = np.ones(data.n_assets) / data.n_assets
ew_ret = float(ew @ data.expected_returns) * 100
ew_vol = float(np.sqrt(ew @ data.cov_matrix @ ew)) * 100
ax.scatter(ew_vol, ew_ret, marker="D", s=100, color="gray", edgecolor="black",
           zorder=10, label="Equal Weight")

# Individual assets
for i, t in enumerate(data.tickers):
    ax.scatter(data.annualized_vols[i] * 100, data.expected_returns[i] * 100,
              s=60, zorder=5, edgecolor="black", color=colors[i])
    ax.annotate(t, (data.annualized_vols[i] * 100, data.expected_returns[i] * 100),
               textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_xlabel("Annualized Volatility (%)")
ax.set_ylabel("Annualized Return (%)")
ax.set_title("Efficient Frontier with Optimal Portfolio", fontweight="bold")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

---
## 4. Risk Parity

Risk Parity equalizes **risk contribution** across assets, solving:

$$\min_w \sum_{i} \left[ \frac{w_i (\Sigma w)_i}{w^T \Sigma w} - \frac{1}{N} \right]^2 \quad \text{s.t.} \quad \sum w_i = 1, \; w_i \geq 0$$

This approach is robust to estimation error in expected returns — it only depends on the covariance matrix.

In [ ]:
rp = RiskParityOptimizer(data)
rp_result = rp.optimize()

print("="*55)
print("       RISK PARITY PORTFOLIO")
print("="*55)
print(f"  Expected Return:  {rp_result.expected_return:>10.2%}")
print(f"  Volatility:       {rp_result.volatility:>10.2%}")
print(f"  Sharpe Ratio:     {rp_result.sharpe_ratio:>10.2f}")
print(f"\n  Weights / Risk Contributions:")
rc = rp_result.extra["risk_contributions"]
for t, w, r in zip(data.tickers, rp_result.weights, rc):
    print(f"    {t:6s}  w={w:>7.1%}   RC={r:>7.1%}")
print("="*55)

In [ ]:
# Risk contributions pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(rp_result.weights, labels=data.tickers, colors=colors,
            autopct="%1.1f%%", startangle=90, pctdistance=0.8)
axes[0].set_title("Risk Parity — Weights", fontweight="bold")

axes[1].pie(rc, labels=data.tickers, colors=colors,
            autopct="%1.1f%%", startangle=90, pctdistance=0.8)
axes[1].set_title("Risk Parity — Risk Contributions", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 5. Black-Litterman

The Black-Litterman model combines **market equilibrium** (implied by market-cap weights) with **analyst views** via Bayes' theorem:

$$\pi = \delta \cdot \Sigma \cdot w_{\text{mkt}}$$

$$\mu_{BL} = \left[(\tau\Sigma)^{-1} + P^T \Omega^{-1} P\right]^{-1} \left[(\tau\Sigma)^{-1} \pi + P^T \Omega^{-1} Q\right]$$

We define two views:
- **View 1 (absolute):** AAPL will return 15% p.a.
- **View 2 (relative):** MSFT will outperform XOM by 5% p.a.

In [ ]:
bl = BlackLittermanOptimizer(data, tau=0.05, risk_aversion=2.5,
                              long_only=True, max_weight=0.40)

# Equilibrium returns (no views)
pi = bl.equilibrium_returns()

# Define views
# View 1: AAPL returns 15%
# View 2: MSFT outperforms XOM by 5%
P = np.array([
    [1, 0, 0, 0, 0],      # AAPL absolute view
    [0, 1, 0, 0, -1],     # MSFT - XOM relative view
])
Q = np.array([0.15, 0.05])

bl_result = bl.optimize(P=P, Q=Q)
mu_bl = bl_result.extra["posterior_returns"]

print("="*60)
print("       BLACK-LITTERMAN PORTFOLIO")
print("="*60)
print(f"  Expected Return:  {bl_result.expected_return:>10.2%}")
print(f"  Volatility:       {bl_result.volatility:>10.2%}")
print(f"  Sharpe Ratio:     {bl_result.sharpe_ratio:>10.2f}")
print(f"\n  Weights:")
for t, w in zip(data.tickers, bl_result.weights):
    print(f"    {t:6s} {w:>8.1%}")
print("="*60)

In [ ]:
# Compare equilibrium vs posterior returns
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(data.tickers))
width = 0.35

bars1 = ax.bar(x - width/2, pi * 100, width, label="Equilibrium (prior)",
               color="#90CAF9", edgecolor="white")
bars2 = ax.bar(x + width/2, mu_bl * 100, width, label="Posterior (with views)",
               color="#1565C0", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(data.tickers)
ax.set_ylabel("Expected Return (%)")
ax.set_title("Black-Litterman: Equilibrium vs Posterior Returns", fontweight="bold")
ax.legend()
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

---
## 6. Walk-Forward Backtest

We run a walk-forward backtest using the max-Sharpe optimizer with:
- **Monthly rebalancing** (21 trading days)
- **1-year lookback** window for estimation (252 days)
- **Ledoit-Wolf shrinkage** at each rebalance
- Benchmark: equal-weight portfolio

In [ ]:
def optimizer_factory(window_data: MarketData) -> dict:
    mv = MeanVarianceOptimizer(window_data, long_only=True, max_weight=0.40)
    result = mv.max_sharpe()
    return {"weights": result.weights}

bt = Backtester(
    optimizer_factory=optimizer_factory,
    rebalance_freq=21,
    lookback_window=252,
    risk_free_rate=data.risk_free_rate,
)

bt_result = bt.run(data)

print(f"Backtest period: {len(bt_result.portfolio_returns)} trading days")
print(f"Rebalance events: {len(bt_result.rebalance_dates)}")
print(f"Average turnover: {np.mean(bt_result.turnover):.2%}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# 1. Equity curves
n_bt = len(bt_result.equity_curve)
axes[0].plot(bt_result.equity_curve, linewidth=2, color="#1565C0", label="Optimized")
axes[0].plot(bt_result.benchmark_curve, linewidth=1.5, color="gray",
             linestyle="--", label="Equal Weight")
for i, t in enumerate(data.tickers):
    axes[0].plot(bt_result.asset_curves[:, i], linewidth=0.8, alpha=0.5, label=t)
axes[0].set_ylabel("Equity (starting = 1.0)")
axes[0].set_title("Walk-Forward Backtest — Equity Curves", fontweight="bold")
axes[0].legend(loc="upper left", fontsize=8, ncol=3)

# 2. Rolling Sharpe
rs = rolling_sharpe(bt_result.portfolio_returns, window=63,
                    rf_daily=data.risk_free_rate / 252)
axes[1].plot(rs, linewidth=1.5, color="#1565C0")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].axhline(bt_result.metrics.sharpe_ratio, color="red", linestyle="--",
                alpha=0.7, label=f"Full-period SR = {bt_result.metrics.sharpe_ratio:.2f}")
axes[1].set_ylabel("Rolling Sharpe (63d)")
axes[1].set_title("Rolling Sharpe Ratio", fontweight="bold")
axes[1].legend()

# 3. Drawdown
dd = drawdown_series(bt_result.equity_curve)
axes[2].fill_between(range(len(dd)), dd * 100, 0, color="#e74c3c", alpha=0.4)
axes[2].plot(dd * 100, color="#e74c3c", linewidth=1)
axes[2].set_ylabel("Drawdown (%)")
axes[2].set_xlabel("Trading Days")
axes[2].set_title(f"Drawdown (max: {bt_result.metrics.max_drawdown:.1%})", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 7. Risk Metrics

We compare three strategies:
- **Optimized**: walk-forward max-Sharpe
- **Equal-weight**: 1/N benchmark
- **Best single asset**: highest Sharpe individual asset (in hindsight)

In [ ]:
# Optimized metrics
opt_metrics = bt_result.metrics.full_report()

# Equal-weight metrics
bm_metrics = bt_result.benchmark_metrics.full_report()

# Best single asset (in-sample)
n_bt_days = len(bt_result.portfolio_returns)
asset_returns = data.returns.values[-n_bt_days:]
best_sharpe = -np.inf
best_idx = 0
for i in range(data.n_assets):
    pm = PortfolioMetrics(asset_returns[:, i], data.risk_free_rate)
    if pm.sharpe_ratio > best_sharpe:
        best_sharpe = pm.sharpe_ratio
        best_idx = i
best_metrics = PortfolioMetrics(asset_returns[:, best_idx], data.risk_free_rate).full_report()
best_name = data.tickers[best_idx]

# Build comparison table
comparison = pd.DataFrame({
    "Optimized": opt_metrics,
    "Equal Weight": bm_metrics,
    f"Best Asset ({best_name})": best_metrics,
}).T

comparison.columns = [
    "Ann. Return", "Ann. Vol", "Sharpe", "Sortino",
    "Max DD", "Calmar", "VaR (95%)", "CVaR (95%)",
]

display(comparison.style.format({
    "Ann. Return": "{:.2%}",
    "Ann. Vol": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Max DD": "{:.2%}",
    "Calmar": "{:.2f}",
    "VaR (95%)": "{:.4f}",
    "CVaR (95%)": "{:.4f}",
}))

---
## Summary

| Method | Key Idea | Pros | Cons |
|--------|----------|------|------|
| **Historical Mean** | $\hat{\mu}_i = \bar{r}_i \cdot 252$ | Simple, no external data | Noisy, overfits to recent history |
| **Fama-French 6-Factor** | Regress on Mkt-RF, SMB, HML, RMW, CMA, Mom | Grounded in factor premia, less noisy | Assumes factors are compensated and stable |
| **Mean-Variance** | Maximize Sharpe ratio | Theoretically optimal | Sensitive to estimation error in returns |
| **Risk Parity** | Equalize risk contributions | No return estimates needed | Ignores expected returns entirely |
| **Black-Litterman** | Bayesian blend of market + views | Stable, intuitive tilts | Requires quality views as input |

### Next Steps
- Add **transaction costs** to the backtester for realistic P&L
- Implement **regime detection** (HMM) for dynamic allocation
- Extend to **multi-asset class** (bonds, commodities, crypto)
- Connect to live portfolio via broker API (Interactive Brokers, Alpaca)